# Overfitting and Generalization

This notebook compares training and validation losses in Python cells, using labeled plots and 95% CI error bars across runs.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "wwgpt").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root; open this notebook from inside the repository.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from wwgpt.analysis import (
    collect_metrics,
    discover_experiment_runs,
    mean_ci95,
    plot_metric_curve,
    terminal_results,
)

RESULTS_ROOT = REPO_ROOT / "results"
ANALYSIS_DIR = RESULTS_ROOT / "notebook_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Results root: {RESULTS_ROOT}")


## Load data in Python


In [ ]:
runs = discover_experiment_runs(RESULTS_ROOT)
metrics = collect_metrics(runs)
terminal = terminal_results(runs)

print(f"Discovered run arms: {len(runs)}")
print(f"Metrics rows loaded: {len(metrics)}")
if terminal.empty:
    display(Markdown("No complete paired terminal validation results were found under `RESULTS_ROOT`."))
else:
    display(terminal)


## Labeled plots with error bars


In [ ]:
plot_specs = [
    ("validation_loss", "validation loss", "validation_loss_by_optimizer.png"),
    ("train_loss", "training loss", "training_loss_by_optimizer.png"),
]
for y_col, ylabel, filename in plot_specs:
    path = plot_metric_curve(
        metrics,
        y_col,
        ANALYSIS_DIR / filename,
        title=f"{ylabel.title()} by optimizer with 95% CI error bars",
        ylabel=ylabel,
    )
    if path is None:
        print(f"Skipping {y_col}: required columns were not available.")
    else:
        display(Markdown(f"Saved `{path}`."))

if not terminal.empty and "wwpgd_minus_adamw_final_validation_loss" in terminal:
    diffs = pd.to_numeric(terminal["wwpgd_minus_adamw_final_validation_loss"], errors="coerce").dropna()
    if len(diffs):
        mean, ci, n = mean_ci95(diffs)
        ax = diffs.plot(kind="bar", figsize=(8, 4), alpha=0.75)
        ax.axhline(0, color="black", linewidth=1)
        ax.errorbar([len(diffs)], [mean], yerr=[ci], fmt="D", color="tab:red", capsize=5, label=f"mean ± 95% CI (n={n})")
        ax.set_xlabel("paired seed index")
        ax.set_ylabel("WW-PGD minus AdamW final validation loss")
        ax.set_title("Paired final validation-loss difference; negative favors WW-PGD")
        ax.grid(True, axis="y", alpha=0.3)
        ax.legend()
        plt.tight_layout()
        out = ANALYSIS_DIR / "paired_final_validation_loss_difference.png"
        plt.savefig(out, dpi=160)
        plt.show()
